In [1]:
%pip install streamlit plotly

Note: you may need to restart the kernel to use updated packages.


In [1]:
%%writefile fraud_visual_dashboard.py

import pandas as pd
import plotly.express as px
import streamlit as st

from pathlib import Path


st.set_page_config(
    page_title="Fraud Visual Analytics",
    page_icon="🛡️",
    layout="wide"
)

BASE_FOLDER = Path(__file__).resolve().parent

TRANSACTION_FILE = (
    BASE_FOLDER /
    "kafka_transaction_results.csv"
)

FRAUD_ALERT_FILE = (
    BASE_FOLDER /
    "kafka_fraud_alerts.csv"
)

MONITORING_FILE = (
    BASE_FOLDER /
    "kafka_monitoring_alerts.csv"
)

ALERT_HISTORY_FILE = (
    BASE_FOLDER /
    "fraud_alert_history.csv"
)

MODEL_METRICS_FILE = (
    BASE_FOLDER /
    "real_time_stream_metrics.csv"
)


def read_csv_safely(file_path):

    if not file_path.exists():
        return pd.DataFrame()

    try:

        return pd.read_csv(file_path)

    except pd.errors.EmptyDataError:

        return pd.DataFrame()

    except Exception as error:

        st.warning(
            f"Could not read {file_path.name}: "
            f"{error}"
        )

        return pd.DataFrame()


@st.cache_data(ttl=5)
def load_dashboard_data():

    transactions = read_csv_safely(
        TRANSACTION_FILE
    )

    fraud_alerts = read_csv_safely(
        FRAUD_ALERT_FILE
    )

    monitoring = read_csv_safely(
        MONITORING_FILE
    )

    history = read_csv_safely(
        ALERT_HISTORY_FILE
    )

    metrics = read_csv_safely(
        MODEL_METRICS_FILE
    )

    return (
        transactions,
        fraud_alerts,
        monitoring,
        history,
        metrics
    )

(
    transaction_data,
    fraud_alerts,
    monitoring_alerts,
    alert_history,
    model_metrics
) = load_dashboard_data()


if not transaction_data.empty:

    numeric_columns = [
        "fraud_probability",
        "risk_score",
        "predicted_fraud"
    ]

    for column in numeric_columns:

        if column in transaction_data.columns:

            transaction_data[column] = (
                pd.to_numeric(
                    transaction_data[column],
                    errors="coerce"
                )
            )

    date_columns = [
        "produced_at",
        "processed_at"
    ]

    for column in date_columns:

        if column in transaction_data.columns:

            transaction_data[column] = (
                pd.to_datetime(
                    transaction_data[column],
                    errors="coerce"
                )
            )

st.sidebar.title(
    "🛡️ Fraud Analytics"
)

selected_page = st.sidebar.radio(
    "Dashboard Page",
    [
        "Executive Overview",
        "Transaction Analytics",
        "Fraud Alerts",
        "Notification Analytics",
        "Model Performance"
    ]
)

if st.sidebar.button(
    "Refresh Data",
    use_container_width=True
):

    st.cache_data.clear()
    st.rerun()

filtered_data = transaction_data.copy()

if not transaction_data.empty:

    if "risk_level" in transaction_data.columns:

        available_levels = (
            transaction_data["risk_level"]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

        risk_order = [
            "Critical",
            "High",
            "Medium",
            "Low"
        ]

        available_levels = [
            level
            for level in risk_order
            if level in available_levels
        ]

        selected_levels = st.sidebar.multiselect(
            "Risk Levels",
            options=available_levels,
            default=available_levels
        )

        filtered_data = filtered_data[
            filtered_data["risk_level"].isin(
                selected_levels
            )
        ]

    if "risk_score" in transaction_data.columns:

        minimum_risk = st.sidebar.slider(
            "Minimum Risk Score",
            min_value=0,
            max_value=100,
            value=0
        )

        filtered_data = filtered_data[
            filtered_data["risk_score"]
            >= minimum_risk
        ]

st.sidebar.markdown("---")

st.sidebar.write(
    "Displayed transactions:",
    len(filtered_data)
)


if selected_page == "Executive Overview":

    st.title(
        "Fraud Detection Executive Dashboard"
    )

    st.caption(
        "Live transaction risk and fraud-monitoring overview"
    )

    if transaction_data.empty:

        st.warning(
            "No Kafka transaction results found. "
            "Run Day 25 producer and consumer first."
        )

    else:

        total_transactions = len(
            filtered_data
        )

        total_fraud = int(
            filtered_data.get(
                "predicted_fraud",
                pd.Series(dtype=float)
            )
            .fillna(0)
            .sum()
        )

        monitoring_cases = int(
            filtered_data.get(
                "risk_level",
                pd.Series(dtype=str)
            )
            .isin(
                [
                    "Medium",
                    "High",
                    "Critical"
                ]
            )
            .sum()
        )

        average_risk = (
            filtered_data["risk_score"].mean()
            if "risk_score" in filtered_data.columns
            else 0
        )

        highest_risk = (
            filtered_data["risk_score"].max()
            if "risk_score" in filtered_data.columns
            else 0
        )

        column1, column2, column3, column4 = (
            st.columns(4)
        )

        column1.metric(
            "Transactions",
            total_transactions
        )

        column2.metric(
            "Fraud Alerts",
            total_fraud
        )

        column3.metric(
            "Monitoring Cases",
            monitoring_cases
        )

        column4.metric(
            "Average Risk",
            f"{average_risk:.2f}"
        )

        st.metric(
            "Highest Risk Score",
            f"{highest_risk:.2f}"
        )

        chart_column1, chart_column2 = (
            st.columns(2)
        )

        with chart_column1:

            st.subheader(
                "Risk-Level Distribution"
            )

            if "risk_level" in filtered_data.columns:

                risk_counts = (
                    filtered_data["risk_level"]
                    .value_counts()
                    .reindex(
                        [
                            "Critical",
                            "High",
                            "Medium",
                            "Low"
                        ],
                        fill_value=0
                    )
                    .reset_index()
                )

                risk_counts.columns = [
                    "risk_level",
                    "transactions"
                ]

                risk_chart = px.pie(
                    risk_counts,
                    names="risk_level",
                    values="transactions",
                    hole=0.45,
                    color="risk_level",
                    color_discrete_map={
                        "Critical": "#8B0000",
                        "High": "#E63946",
                        "Medium": "#F4A261",
                        "Low": "#2A9D8F"
                    }
                )

                st.plotly_chart(
                    risk_chart,
                    use_container_width=True
                )

        with chart_column2:

            st.subheader(
                "Fraud Prediction Distribution"
            )

            if "predicted_fraud" in filtered_data.columns:

                prediction_counts = (
                    filtered_data[
                        "predicted_fraud"
                    ]
                    .map(
                        {
                            0: "Normal",
                            1: "Fraud"
                        }
                    )
                    .value_counts()
                    .reset_index()
                )

                prediction_counts.columns = [
                    "prediction",
                    "transactions"
                ]

                prediction_chart = px.bar(
                    prediction_counts,
                    x="prediction",
                    y="transactions",
                    color="prediction",
                    color_discrete_map={
                        "Normal": "#2A9D8F",
                        "Fraud": "#E63946"
                    },
                    text_auto=True
                )

                st.plotly_chart(
                    prediction_chart,
                    use_container_width=True
                )

        st.subheader(
            "Top-Risk Transactions"
        )

        if "risk_score" in filtered_data.columns:

            highest_risk_data = (
                filtered_data
                .sort_values(
                    by="risk_score",
                    ascending=False
                )
                .head(10)
            )

            st.dataframe(
                highest_risk_data,
                use_container_width=True,
                hide_index=True
            )



elif selected_page == "Transaction Analytics":

    st.title(
        "Transaction Visual Analytics"
    )

    if filtered_data.empty:

        st.info(
            "No transactions match the selected filters."
        )

    else:

        chart_column1, chart_column2 = (
            st.columns(2)
        )

        with chart_column1:

            st.subheader(
                "Risk-Score Distribution"
            )

            if "risk_score" in filtered_data.columns:

                risk_histogram = px.histogram(
                    filtered_data,
                    x="risk_score",
                    nbins=20,
                    color="risk_level"
                    if "risk_level"
                    in filtered_data.columns
                    else None,
                    title="Transaction Risk Scores"
                )

                st.plotly_chart(
                    risk_histogram,
                    use_container_width=True
                )

        with chart_column2:

            st.subheader(
                "Fraud-Probability Distribution"
            )

            if (
                "fraud_probability"
                in filtered_data.columns
            ):

                probability_histogram = (
                    px.histogram(
                        filtered_data,
                        x="fraud_probability",
                        nbins=20,
                        color="risk_level"
                        if "risk_level"
                        in filtered_data.columns
                        else None,
                        title=(
                            "Predicted Fraud Probabilities"
                        )
                    )
                )

                st.plotly_chart(
                    probability_histogram,
                    use_container_width=True
                )

        if all(
            column in filtered_data.columns
            for column in [
                "fraud_probability",
                "risk_score",
                "risk_level"
            ]
        ):

            st.subheader(
                "Probability and Risk Relationship"
            )

            scatter_chart = px.scatter(
                filtered_data,
                x="fraud_probability",
                y="risk_score",
                color="risk_level",
                hover_data=[
                    "stream_id"
                ]
                if "stream_id"
                in filtered_data.columns
                else None,
                title=(
                    "Fraud Probability vs Risk Score"
                )
            )

            st.plotly_chart(
                scatter_chart,
                use_container_width=True
            )

        if (
            "processed_at" in filtered_data.columns
            and filtered_data[
                "processed_at"
            ].notna().any()
        ):

            st.subheader(
                "Risk Score Over Time"
            )

            timeline_data = (
                filtered_data
                .dropna(
                    subset=["processed_at"]
                )
                .sort_values(
                    by="processed_at"
                )
            )

            timeline_chart = px.line(
                timeline_data,
                x="processed_at",
                y="risk_score",
                markers=True,
                title="Streaming Transaction Risk"
            )

            st.plotly_chart(
                timeline_chart,
                use_container_width=True
            )

        st.subheader(
            "Filtered Transaction Table"
        )

        st.dataframe(
            filtered_data,
            use_container_width=True,
            hide_index=True
        )

        st.download_button(
            "Download Filtered Transactions",
            filtered_data.to_csv(
                index=False
            ),
            "filtered_transactions.csv",
            "text/csv"
        )


elif selected_page == "Fraud Alerts":

    st.title(
        "Fraud Alerts and Investigation Queue"
    )

    fraud_tab, monitoring_tab = st.tabs(
        [
            "Confirmed Fraud Alerts",
            "Monitoring Cases"
        ]
    )

    with fraud_tab:

        if fraud_alerts.empty:

            st.info(
                "No confirmed fraud alerts are available."
            )

        else:

            st.metric(
                "Confirmed Fraud Alerts",
                len(fraud_alerts)
            )

            if "risk_score" in fraud_alerts.columns:

                fraud_alerts = fraud_alerts.sort_values(
                    by="risk_score",
                    ascending=False
                )

            st.dataframe(
                fraud_alerts,
                use_container_width=True,
                hide_index=True
            )

    with monitoring_tab:

        if monitoring_alerts.empty:

            st.info(
                "No monitoring cases are available."
            )

        else:

            st.metric(
                "Monitoring Cases",
                len(monitoring_alerts)
            )

            if (
                "risk_level"
                in monitoring_alerts.columns
            ):

                monitoring_chart_data = (
                    monitoring_alerts[
                        "risk_level"
                    ]
                    .value_counts()
                    .reset_index()
                )

                monitoring_chart_data.columns = [
                    "risk_level",
                    "transactions"
                ]

                monitoring_chart = px.bar(
                    monitoring_chart_data,
                    x="risk_level",
                    y="transactions",
                    color="risk_level",
                    text_auto=True
                )

                st.plotly_chart(
                    monitoring_chart,
                    use_container_width=True
                )

            st.dataframe(
                monitoring_alerts,
                use_container_width=True,
                hide_index=True
            )


elif selected_page == "Notification Analytics":

    st.title(
        "Automated Alert Notification Analytics"
    )

    if alert_history.empty:

        st.info(
            "No alert history is available."
        )

    else:

        total_notifications = len(
            alert_history
        )

        unique_transactions = (
            alert_history["stream_id"].nunique()
            if "stream_id"
            in alert_history.columns
            else total_notifications
        )

        notification_column1, notification_column2 = (
            st.columns(2)
        )

        notification_column1.metric(
            "Total Notifications",
            total_notifications
        )

        notification_column2.metric(
            "Unique Transactions",
            unique_transactions
        )

        chart_column1, chart_column2 = (
            st.columns(2)
        )

        with chart_column1:

            if "risk_level" in alert_history.columns:

                risk_notification_data = (
                    alert_history["risk_level"]
                    .value_counts()
                    .reset_index()
                )

                risk_notification_data.columns = [
                    "risk_level",
                    "notifications"
                ]

                risk_notification_chart = px.bar(
                    risk_notification_data,
                    x="risk_level",
                    y="notifications",
                    color="risk_level",
                    text_auto=True,
                    title="Notifications by Risk"
                )

                st.plotly_chart(
                    risk_notification_chart,
                    use_container_width=True
                )

        with chart_column2:

            if (
                "notification_status"
                in alert_history.columns
            ):

                status_data = (
                    alert_history[
                        "notification_status"
                    ]
                    .value_counts()
                    .reset_index()
                )

                status_data.columns = [
                    "status",
                    "notifications"
                ]

                status_chart = px.pie(
                    status_data,
                    names="status",
                    values="notifications",
                    hole=0.40,
                    title="Notification Status"
                )

                st.plotly_chart(
                    status_chart,
                    use_container_width=True
                )

        st.dataframe(
            alert_history,
            use_container_width=True,
            hide_index=True
        )



elif selected_page == "Model Performance":

    st.title(
        "Fraud Detection Model Performance"
    )

    if model_metrics.empty:

        st.info(
            "The model metrics file is unavailable."
        )

    elif all(
        column in model_metrics.columns
        for column in [
            "metric",
            "value"
        ]
    ):

        model_metrics["value"] = pd.to_numeric(
            model_metrics["value"],
            errors="coerce"
        )

        metric_columns = st.columns(
            len(model_metrics)
        )

        for metric_column, (_, row) in zip(
            metric_columns,
            model_metrics.iterrows()
        ):

            metric_column.metric(
                row["metric"],
                f"{row['value']:.4f}"
            )

        metric_chart = px.bar(
            model_metrics,
            x="metric",
            y="value",
            color="metric",
            text_auto=".4f",
            title="Model Metric Comparison"
        )

        metric_chart.update_yaxes(
            range=[0, 1]
        )

        st.plotly_chart(
            metric_chart,
            use_container_width=True
        )

        st.dataframe(
            model_metrics,
            use_container_width=True,
            hide_index=True
        )

    else:

        st.dataframe(
            model_metrics,
            use_container_width=True,
            hide_index=True
        )



st.markdown("---")

st.caption(
    "Financial Fraud Detection, Risk Scoring "
    "and Real-Time Monitoring System"
)

Overwriting fraud_visual_dashboard.py


In [2]:
from pathlib import Path

app_file = Path(
    "fraud_visual_dashboard.py"
)

print(
    "Dashboard created:",
    app_file.exists()
)

print(
    "File size:",
    app_file.stat().st_size
    if app_file.exists()
    else 0
)

Dashboard created: True
File size: 20504


In [3]:
import subprocess
import sys

syntax_check = subprocess.run(
    [
        sys.executable,
        "-m",
        "py_compile",
        "fraud_visual_dashboard.py"
    ],
    capture_output=True,
    text=True
)

if syntax_check.returncode == 0:
    print("No syntax or indentation errors found!")
else:
    print(syntax_check.stderr)

No syntax or indentation errors found!


In [4]:
import subprocess
import sys

dashboard_process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        "fraud_visual_dashboard.py"
    ]
)

print("Open: http://localhost:8501")

Open: http://localhost:8501


In [ ]:
#to stop it later 
dashboard_process.terminate()